# Combine Chromosomes

In [1]:
%%bash

#!/bin/bash

set -o pipefail
set -o errexit

# Grab existing run info
source "/home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/plink_results/latest_run.env"

echo "Combining outputs from ${OUTPUT_PATH}"

PGS_TAG="${PGS// /_}"
log_file="${OUTPUT_PATH}/combine_chr_${PGS_TAG}_${RUN_ID}_log.txt"
exec > >(tee -a "${log_file}") 2>&1

read -r -a PGS_IDs <<< "${PGS}"

for PGS_ID in "${PGS_IDs[@]}"; do
    echo "Running ${PGS_ID}"
    combined_file="${OUTPUT_PATH}/${PGS_ID}_combined_chr1_22_${RUN_ID}.csv"
    contributions_file="${OUTPUT_PATH}/${PGS_ID}_chr_contributions_${RUN_ID}.tsv"
    : > "${contributions_file}"

    for chromo in {1..22} X; do
        sscore_file="${OUTPUT_PATH}/scores_chr${chromo}.sscore"

        if [[ ! -s "${sscore_file}" ]]; then
            echo "Skipping chr${chromo}: score file not found: ${sscore_file}"
            continue
        fi

        # Extract this PGS's raw score sum, keyed by family and individual ID.
    awk -F '\t' -v pgs="${PGS_ID}" '
        BEGIN { OFS = "\t" }
        FNR == 1 {
            for (i = 1; i <= NF; i++) {
                name = $i
                sub(/^#/, "", name)
                if (name == "FID") fid_col = i
                if (name == "IID") iid_col = i
                if (name == pgs "_SUM") score_col = i
            }
            if (!fid_col || !iid_col || !score_col) {
                print "Required FID, IID, or " pgs "_SUM column missing" > "/dev/stderr"
                exit 2
            }
            next
        }
        { print $fid_col, $iid_col, $score_col }
    ' "${sscore_file}" >> "${contributions_file}"

    done

    # Sum per-chromosome raw scores for each sample.
    awk -F '\t' '
        BEGIN {
            OFS = ","
        }
        {
            key = $1 SUBSEP $2
            fid[key] = $1
            iid[key] = $2
            score[key] += $3
        }
        END {
            print "FID", "IID", "SCORE_SUM"
            for (key in score)
                print fid[key], iid[key], score[key]
        }
    ' "${contributions_file}" > "${combined_file}"
    rm -f "${contributions_file}"

    echo "Combined file written to ${combined_file}"
done

Combining outputs from /home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/plink_results/PGS002308_PGS003446_GRCh38_20261123_151336
Running PGS002308
Skipping chr1: score file not found: /home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/plink_results/PGS002308_PGS003446_GRCh38_20261123_151336/scores_chr1.sscore
Skipping chr2: score file not found: /home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/plink_results/PGS002308_PGS003446_GRCh38_20261123_151336/scores_chr2.sscore
Skipping chr3: score file not found: /home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/plink_results/PGS002308_PGS003446_GRCh38_20261123_151336/scores_chr3.sscore
Skipping chr4: score file not found: /home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/plink_results/PGS002308_PGS003446_GRCh38_20261123_151336/scores_chr4.sscore
Skipping chr5: score file not found: /home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/plink_results/PGS002308_PGS003446_GRCh38_20261123_151336/sc

# Check Variant Incorperation Fraction

In [2]:
import csv
import gzip
import shlex
from pathlib import Path

# Read run settings written by the scoring script.
run_info = Path("/home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/""plink_results/latest_run.env")
run_settings = {}
for line in run_info.read_text().splitlines():
    key, _, value = line.partition("=")
    parsed = shlex.split(value)
    run_settings[key] = parsed[0] if parsed else ""

OUTPUT_PATH = Path(run_settings["OUTPUT_PATH"])
RUN_ID = run_settings["RUN_ID"]
PGS_IDs = run_settings["PGS"].split()

pgs_ids_tag = "_".join(PGS_IDs)
BASE_DIRECTORY = Path("/home/jupyter/workspace/workspace-bucket/calculate_pgs_bed")
WEIGHTS_FILE = (BASE_DIRECTORY/ "pgs_bim_matched_weights"/ f"combined_plink_score_{pgs_ids_tag}_GRCh38_wide.txt.gz")

chromosomes = [str(i) for i in range(1, 23)] + ["X"] + ["Y"]

# Read the variant IDs PLINK used for each chromosome.
used_ids = {}
for chrom in chromosomes:
    vars_file = OUTPUT_PATH / f"scores_chr{chrom}.sscore.vars"
    if vars_file.exists():
        used_ids[chrom] = {
            line.split()[0]
            for line in vars_file.read_text().splitlines()
            if line.strip()
        }
    else:
        used_ids[chrom] = set()

counts = {
    pgs_id: {chrom: {"possible": 0, "used": 0} for chrom in chromosomes}
    for pgs_id in PGS_IDs
}

with gzip.open(WEIGHTS_FILE, "rt", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
  
    for row in reader:
        snp_id = row["SNP_ID"]
        chrom = snp_id.split(":", 1)[0]
        if chrom.lower().startswith("chr"):
            chrom = chrom[3:]
        chrom = chrom.upper()

        if chrom not in chromosomes:
            continue

        for pgs_id in PGS_IDs:
            value = row[pgs_id]
            weight = float(value) if value not in (None, "") else 0.0
            if weight == 0:
                continue

            counts[pgs_id][chrom]["possible"] += 1
            if snp_id in used_ids[chrom]:
                counts[pgs_id][chrom]["used"] += 1

percentages = {}

for pgs_id in PGS_IDs:
    summary_file = OUTPUT_PATH / (f"variants_incorporated_by_chr_{pgs_id}_{RUN_ID}.csv")

    possible_total = sum(counts[pgs_id][chrom]["possible"] for chrom in chromosomes)
    used_total = sum(counts[pgs_id][chrom]["used"] for chrom in chromosomes)

    with summary_file.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "chromosome",
            "total_possible",
            "valid_predictors_loaded"
        ])

        for chrom in chromosomes:
            writer.writerow([
                chrom,
                counts[pgs_id][chrom]["possible"],
                counts[pgs_id][chrom]["used"]
            ])

        writer.writerow(["TOTAL", possible_total, used_total])

    percentages[pgs_id] = (
        f"{100 * used_total / possible_total:.2f}"
        if possible_total
        else "NA"
    )

    print(
        f"{pgs_id}: possible={possible_total:,}, used={used_total:,}, "
        f"incorporated={percentages[pgs_id]}%"
    )
    print(f"Wrote {summary_file}")

percent_file = OUTPUT_PATH / (f"percent_incorporated_{'_'.join(PGS_IDs)}_{RUN_ID}.csv")

with percent_file.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", *PGS_IDs])
    writer.writerow(["percent_incorporated", *[
        percentages[pgs_id] for pgs_id in PGS_IDs
    ]])

print(f"Wrote {percent_file}")

PGS002308: possible=1,258,327, used=816,520, incorporated=64.89%
Wrote /home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/plink_results/PGS002308_PGS003446_GRCh38_20261123_151336/variants_incorporated_by_chr_PGS002308_20261123_151336.csv
PGS003446: possible=520,016, used=339,469, incorporated=65.28%
Wrote /home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/plink_results/PGS002308_PGS003446_GRCh38_20261123_151336/variants_incorporated_by_chr_PGS003446_20261123_151336.csv
Wrote /home/jupyter/workspace/workspace-bucket/calculate_pgs_bed/plink_results/PGS002308_PGS003446_GRCh38_20261123_151336/percent_incorporated_PGS002308_PGS003446_20261123_151336.csv
